# Generating segmentation predictions

As the upcoming model is used to refine segmentations... # TODO

In [3]:
from utils.device import get_device

device = get_device()

PyTorch version: 2.7.0+cu128
is cuda available: True


In [ ]:
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["FIVES"]

train_split = dataset_choice.preferred_train_split

## Loading the model

### Case #1: Use the provided weights (see [README](../README.md)) to initialize the pretrained U-Net model
> Warning: The provided weights are from a model trained on the FIVES dataset, for DRIVE or another dataset, you must use the U-Net model trained in the preceding notebook (see Case #2)

In [5]:
from image_segmentation.models import BinarySegmentator

provided_ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretrained")

model = BinarySegmentator.load_from_checkpoint(provided_ckpt_path, map_location=device)

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)


### Case #2: Use the U-Net model trained in the [preceding notebook (2)](./02_pretrain_unet.ipynb)

In [ ]:
from image_segmentation.models import BinarySegmentator

ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretraining")

model = BinarySegmentator.load_from_checkpoint(ckpt_path, map_location=device)



### Case #3: Use your own model. In this case you will need to adapt the code below to load your model and its weights, and to use it for generating the binary predictions.

### Case #4: You already have precomputed predictions, or want to use the path classification model on ground truths, you don't need to execute the following cells of this notebook and can directly go to the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)

## Generating the prediction masks

In [ ]:
from image_segmentation.data import ImageDataset
from image_segmentation.data.augmentations import build_val_transform

data_dir = dataset_choice.get_data_dir()
dataset = ImageDataset(data_dir, transforms=None)

stats = dataset.get_dataset_stats(split_name=train_split)
mean = stats["foreground"]["mean"]
std = stats["foreground"]["std"]

test_transforms = build_val_transform(mean=mean, std=std)
dataset.transforms = test_transforms

In [ ]:
import gc
import os
import torch
from PIL import Image
from tqdm import tqdm

pred_dir = os.path.join(data_dir, 'pred')
os.makedirs(pred_dir, exist_ok=True)

model.eval()
model.to(device)

with torch.no_grad():
    for i, img_path in enumerate(tqdm(dataset.img_paths)):
        img, _= dataset[i]  # torch.Tensor, shape (H, W, C)
        img_tensor = img.unsqueeze(0).float().to(device)

        logits = model(img_tensor)
        probs = torch.sigmoid(logits)
        pred = probs.squeeze().cpu().numpy()  # (H, W)

        pred_bin = (pred > 0.5).astype('uint8')

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        pred_path = os.path.join(pred_dir, base_name + '.png')
        Image.fromarray(pred_bin * 255).save(pred_path)

        del img_tensor, logits, probs, pred, pred_bin, img

torch.cuda.empty_cache()
gc.collect()

In [ ]:
print(f"Predictions saved to {pred_dir}, total {len(os.listdir(pred_dir))} images.")

Now that we have prediction masks to execute the main model on, you can continue on the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)